# Bitcoin (BTC) Price Prediction using LSTM

This notebook implements a deep learning model using LSTM (Long Short-Term Memory) to predict Bitcoin prices.

**Dataset:**
- Training: Historical data until December 31, 2024
- Testing: Data from January 1, 2025 onwards

**Model Architecture:**
- 3-layer stacked LSTM with Dropout regularization

---
## 1. Import Libraries

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Data fetching
import yfinance as yf

# Preprocessing
from sklearn.preprocessing import MinMaxScaler

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---
## 2. Data Collection

Fetching Bitcoin (BTC-USD) historical data from Yahoo Finance.

In [ ]:
# Define the ticker symbol
TICKER = 'BTC-USD'

# Fetch data from the beginning until today
print(f"Fetching {TICKER} data from Yahoo Finance...")
df = yf.download(TICKER, start='2014-01-01', end='2025-12-31', progress=False)

# Reset index to make Date a column
df = df.reset_index()

# Display basic info
print(f"\n{'='*50}")
print(f"Dataset Information")
print(f"{'='*50}")
print(f"Data Range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Total Records: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Display last 5 rows
print("Last 5 rows:")
df.tail()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Visualize the historical price
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], color='#2196F3', linewidth=1)
plt.title('Bitcoin (BTC) Historical Price', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.show()

---
## 3. Data Preprocessing

- Check for missing values
- Normalize the data using MinMaxScaler
- Create sequences for LSTM input

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Handle missing values if any
if df.isnull().sum().sum() > 0:
    print("\nHandling missing values...")
    df = df.fillna(method='ffill')
    df = df.fillna(method='bfill')
    print("Missing values filled using forward/backward fill.")
else:
    print("\nNo missing values found!")

In [ ]:
# We will use 'Close' price for prediction
# Extract the Close price column
close_prices = df[['Date', 'Close']].copy()
close_prices.set_index('Date', inplace=True)

print(f"Close Price Data Shape: {close_prices.shape}")
close_prices.head()

In [ ]:
# Normalize the data using MinMaxScaler (scale to 0-1 range)
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit and transform the data
scaled_data = scaler.fit_transform(close_prices.values)

print(f"Scaled Data Shape: {scaled_data.shape}")
print(f"Min Value: {scaled_data.min():.4f}")
print(f"Max Value: {scaled_data.max():.4f}")

In [ ]:
# Function to create sequences for LSTM
def create_sequences(data, seq_length):
    """
    Create sequences of data for LSTM input.
    
    Args:
        data: Scaled data array
        seq_length: Number of time steps to look back
    
    Returns:
        X: Feature sequences
        y: Target values
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Define sequence length (look-back period)
SEQUENCE_LENGTH = 60  # 60 days of historical data to predict next day

print(f"Sequence Length: {SEQUENCE_LENGTH} days")

---
## 4. Train/Test Split

- **Training Data**: From the beginning until December 31, 2024
- **Testing Data**: From January 1, 2025 onwards

In [ ]:
# Define the split date
SPLIT_DATE = '2024-12-31'

# Get the index of the split date
split_idx = close_prices.index.get_indexer([pd.Timestamp(SPLIT_DATE)], method='nearest')[0]

print(f"Split Date: {SPLIT_DATE}")
print(f"Split Index: {split_idx}")
print(f"Training Period: {close_prices.index[0].strftime('%Y-%m-%d')} to {close_prices.index[split_idx].strftime('%Y-%m-%d')}")
print(f"Testing Period: {close_prices.index[split_idx+1].strftime('%Y-%m-%d')} to {close_prices.index[-1].strftime('%Y-%m-%d')}")

In [ ]:
# Split the scaled data
train_data = scaled_data[:split_idx + 1]
test_data = scaled_data[split_idx + 1 - SEQUENCE_LENGTH:]  # Include sequence length for context

print(f"Training Data Shape: {train_data.shape}")
print(f"Testing Data Shape: {test_data.shape}")

In [ ]:
# Create sequences for training
X_train, y_train = create_sequences(train_data, SEQUENCE_LENGTH)

# Create sequences for testing
X_test, y_test = create_sequences(test_data, SEQUENCE_LENGTH)

print(f"X_train Shape: {X_train.shape}")
print(f"y_train Shape: {y_train.shape}")
print(f"X_test Shape: {X_test.shape}")
print(f"y_test Shape: {y_test.shape}")

In [ ]:
# Reshape data for LSTM input [samples, time steps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f"\nAfter reshaping for LSTM:")
print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")

---
## 5. LSTM Model Architecture

Building a 3-layer stacked LSTM model with Dropout regularization to prevent overfitting.

In [ ]:
# Build the LSTM model
def build_lstm_model(input_shape):
    """
    Build a stacked LSTM model for time series prediction.
    
    Architecture:
    - LSTM Layer 1: 50 units, return sequences
    - Dropout: 20%
    - LSTM Layer 2: 50 units, return sequences
    - Dropout: 20%
    - LSTM Layer 3: 50 units
    - Dropout: 20%
    - Dense Output: 1 unit
    """
    model = Sequential([
        # First LSTM layer
        LSTM(units=50, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        
        # Second LSTM layer
        LSTM(units=50, return_sequences=True),
        Dropout(0.2),
        
        # Third LSTM layer
        LSTM(units=50, return_sequences=False),
        Dropout(0.2),
        
        # Output layer
        Dense(units=1)
    ])
    
    return model

# Create the model
model = build_lstm_model((X_train.shape[1], 1))

# Compile the model
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

# Display model summary
print("\nModel Summary:")
model.summary()

---
## 6. Model Training

Training the LSTM model with Early Stopping to prevent overfitting.

In [ ]:
# Define training parameters
EPOCHS = 50
BATCH_SIZE = 32

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

print(f"Training Parameters:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Early Stopping Patience: 10")
print(f"\nStarting training...")

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1
)

print(f"\nTraining completed!")
print(f"Final Training Loss: {history.history['loss'][-1]:.6f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.6f}")

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))

plt.plot(history.history['loss'], label='Training Loss', color='#2196F3', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='#FF5722', linewidth=2)

plt.title('Model Training History', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Prediction & Evaluation

Making predictions on the test data (2025) and evaluating model performance.

In [ ]:
# Make predictions on test data
print("Making predictions on test data (2025)...")
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse transform predictions and actual values
y_pred = scaler.inverse_transform(y_pred_scaled)
y_actual = scaler.inverse_transform(y_test)

print(f"Predictions Shape: {y_pred.shape}")
print(f"Actual Values Shape: {y_actual.shape}")

In [ ]:
# Calculate evaluation metrics
def calculate_metrics(y_true, y_pred):
    """
    Calculate RMSE, MAE, and MAPE.
    """
    # RMSE (Root Mean Squared Error)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # MAE (Mean Absolute Error)
    mae = mean_absolute_error(y_true, y_pred)
    
    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return rmse, mae, mape

# Calculate metrics
rmse, mae, mape = calculate_metrics(y_actual, y_pred)

print(f"\n{'='*50}")
print(f"Model Evaluation Metrics (Test Data - 2025)")
print(f"{'='*50}")
print(f"RMSE (Root Mean Squared Error): ${rmse:,.2f}")
print(f"MAE  (Mean Absolute Error):     ${mae:,.2f}")
print(f"MAPE (Mean Absolute % Error):   {mape:.2f}%")
print(f"{'='*50}")

In [ ]:
# Create a DataFrame with actual vs predicted values
test_dates = close_prices.index[split_idx + 1:split_idx + 1 + len(y_pred)]

results_df = pd.DataFrame({
    'Date': test_dates,
    'Actual': y_actual.flatten(),
    'Predicted': y_pred.flatten()
})
results_df['Difference'] = results_df['Actual'] - results_df['Predicted']
results_df['Percentage_Error'] = (abs(results_df['Difference']) / results_df['Actual']) * 100

print("\nPrediction Results (2025):")
results_df.head(10)

In [ ]:
# Display tail of results
print("\nLast 10 predictions:")
results_df.tail(10)

---
## 8. Visualization

Visualizing the prediction results.

In [ ]:
# Plot Actual vs Predicted prices for 2025
plt.figure(figsize=(14, 7))

plt.plot(results_df['Date'], results_df['Actual'], 
         label='Actual Price', color='#2196F3', linewidth=2)
plt.plot(results_df['Date'], results_df['Predicted'], 
         label='Predicted Price', color='#FF5722', linewidth=2, linestyle='--')

plt.title('Bitcoin Price Prediction - 2025 (LSTM Model)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot complete price history with prediction
plt.figure(figsize=(16, 8))

# Training period
train_dates = close_prices.index[:split_idx + 1]
train_prices = close_prices.values[:split_idx + 1]

plt.plot(train_dates, train_prices, label='Training Data', color='#4CAF50', linewidth=1, alpha=0.7)
plt.plot(results_df['Date'], results_df['Actual'], label='Actual (2025)', color='#2196F3', linewidth=2)
plt.plot(results_df['Date'], results_df['Predicted'], label='Predicted (2025)', color='#FF5722', linewidth=2, linestyle='--')

# Add vertical line at split date
plt.axvline(x=pd.Timestamp(SPLIT_DATE), color='red', linestyle=':', linewidth=2, label='Train/Test Split')

plt.title('Bitcoin Price: Historical Data & 2025 Prediction', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Prediction Error Distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(results_df['Difference'], bins=30, color='#9C27B0', edgecolor='white', alpha=0.7)
plt.title('Prediction Error Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Error (Actual - Predicted)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(results_df['Percentage_Error'], bins=30, color='#00BCD4', edgecolor='white', alpha=0.7)
plt.title('Percentage Error Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Percentage Error (%)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics
print(f"\n{'='*60}")
print(f"SUMMARY - Bitcoin Price Prediction using LSTM")
print(f"{'='*60}")
print(f"\nDataset:")
print(f"  - Training Period: {close_prices.index[0].strftime('%Y-%m-%d')} to {SPLIT_DATE}")
print(f"  - Testing Period:  2025-01-01 to {results_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"  - Sequence Length: {SEQUENCE_LENGTH} days")
print(f"\nModel Architecture:")
print(f"  - 3 LSTM layers (50 units each)")
print(f"  - Dropout: 20%")
print(f"  - Optimizer: Adam")
print(f"  - Loss: Mean Squared Error")
print(f"\nEvaluation Metrics (2025 Test Data):")
print(f"  - RMSE: ${rmse:,.2f}")
print(f"  - MAE:  ${mae:,.2f}")
print(f"  - MAPE: {mape:.2f}%")
print(f"{'='*60}")